In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

file_path = r"C:\Customer_Revenue_Intelligence\data\raw\online_retail_II.xlsx"

df_1 = pd.read_excel(file_path, sheet_name="Year 2009-2010", engine="openpyxl")
df_2 = pd.read_excel(file_path, sheet_name="Year 2010-2011", engine="openpyxl")
df = pd.concat([df_1, df_2], ignore_index=True)

print(f"Raw dataset loaded: {len(df):,} rows, {df.shape[1]} columns")

Raw dataset loaded: 1,067,371 rows, 8 columns


In [2]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Before: {before:,} rows")
print(f"Removed: {before - after:,} duplicate rows")
print(f"After:  {after:,} rows")

Before: 1,067,371 rows
Removed: 34,335 duplicate rows
After:  1,033,036 rows


In [3]:
before = len(df)
df = df[~df['Invoice'].astype(str).str.startswith('C')]
after = len(df)

print(f"Before: {before:,} rows")
print(f"Removed: {before - after:,} cancellation rows (C-invoice)")
print(f"After:  {after:,} rows")

Before: 1,033,036 rows
Removed: 19,104 cancellation rows (C-invoice)
After:  1,013,932 rows


In [4]:
before = len(df)
df = df[~((df['Quantity'] < 0) & (~df['Invoice'].astype(str).str.startswith('C')))]
after = len(df)

print(f"Before: {before:,} rows")
print(f"Removed: {before - after:,} stock adjustment rows")
print(f"After:  {after:,} rows")

Before: 1,013,932 rows
Removed: 3,393 stock adjustment rows
After:  1,010,539 rows


In [5]:
before = len(df)
df = df[df['Price'] > 0]
after = len(df)

print(f"Before: {before:,} rows")
print(f"Removed: {before - after:,} zero/negative price rows")
print(f"After:  {after:,} rows")

Before: 1,010,539 rows
Removed: 2,626 zero/negative price rows
After:  1,007,913 rows


In [6]:
before = len(df)
df_clean = df[df['Customer ID'].notnull()].copy()
after = len(df_clean)

print(f"Before: {before:,} rows")
print(f"Removed: {before - after:,} null Customer ID rows")
print(f"After:  {after:,} rows")
print(f"\ndf_clean is our final working dataset for customer-level analysis")

Before: 1,007,913 rows
Removed: 228,488 null Customer ID rows
After:  779,425 rows

df_clean is our final working dataset for customer-level analysis


In [7]:
before_dtype = df_clean['Customer ID'].dtype

df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

print(f"Customer ID dtype before: {before_dtype}")
print(f"Customer ID dtype after:  {df_clean['Customer ID'].dtype}")
print(f"\nSample Customer IDs after conversion:")
print(df_clean['Customer ID'].head(5).tolist())

Customer ID dtype before: float64
Customer ID dtype after:  int32

Sample Customer IDs after conversion:
[13085, 13085, 13085, 13085, 13085]


In [8]:
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

print(f"Revenue column created successfully")
print(f"\nSample Revenue values:")
print(df_clean[['Invoice', 'Quantity', 'Price', 'Revenue']].head(5).to_string())
print(f"\nRevenue statistics:")
print(f"Min Revenue:    £{df_clean['Revenue'].min():,.2f}")
print(f"Max Revenue:    £{df_clean['Revenue'].max():,.2f}")
print(f"Mean Revenue:   £{df_clean['Revenue'].mean():,.2f}")
print(f"Total Revenue:  £{df_clean['Revenue'].sum():,.2f}")

Revenue column created successfully

Sample Revenue values:
  Invoice  Quantity  Price  Revenue
0  489434        12   6.95     83.4
1  489434        12   6.75     81.0
2  489434        12   6.75     81.0
3  489434        48   2.10    100.8
4  489434        24   1.25     30.0

Revenue statistics:
Min Revenue:    £0.00
Max Revenue:    £168,469.60
Mean Revenue:   £22.29
Total Revenue:  £17,374,804.27


In [9]:
zero_revenue = df_clean[df_clean['Revenue'] == 0]
print(f"Zero revenue rows: {len(zero_revenue):,}")
print(f"\nSample zero revenue rows:")
print(zero_revenue[['Invoice','StockCode','Description','Quantity','Price','Revenue','Customer ID']].head(10).to_string())

Zero revenue rows: 0

Sample zero revenue rows:
Empty DataFrame
Columns: [Invoice, StockCode, Description, Quantity, Price, Revenue, Customer ID]
Index: []


In [10]:
print(f"Exact minimum Revenue value: {df_clean['Revenue'].min()}")
print(f"Rows with Revenue under £0.01: {len(df_clean[df_clean['Revenue'] < 0.01]):,}")
print(f"Rows with Revenue under £0.10: {len(df_clean[df_clean['Revenue'] < 0.10]):,}")
print(f"\nSample of lowest revenue rows:")
print(df_clean.nsmallest(5, 'Revenue')[['Invoice','Quantity','Price','Revenue']].to_string())

Exact minimum Revenue value: 0.001
Rows with Revenue under £0.01: 18
Rows with Revenue under £0.10: 23

Sample of lowest revenue rows:
      Invoice  Quantity  Price  Revenue
62299  494914         1  0.001    0.001
74731  496222         1  0.001    0.001
77702  496473         1  0.001    0.001
79794  496643         1  0.001    0.001
90798  497935         1  0.001    0.001


In [ ]:
print(f"Actual minimum revenue (unrounded): {df_clean['Revenue'].min()}")
print(f"Rows with Revenue under £0.01: {len(df_clean[df_clean['Revenue'] < 0.01]):,}")

Actual minimum revenue (unrounded): 0.001
Rows with Revenue under £0.01: 18


In [12]:
tiny = df_clean[df_clean['Revenue'] < 0.01]
print(tiny[['Invoice','StockCode','Description','Quantity','Price','Revenue']].to_string())

       Invoice     StockCode                 Description  Quantity  Price  Revenue
62299   494914          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
74731   496222          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
77702   496473          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
79794   496643          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
90798   497935          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
97716   498562          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
101718  499056          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
104480  499399          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
123947  501176          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
156809  504332          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
178629  506303          PADS  PADS TO MATCH ALL CUSHIONS         1  0.001    0.001
3418

In [13]:
before = len(df_clean)
df_clean = df_clean[df_clean['Revenue'] >= 0.01]
after = len(df_clean)

print(f"Before: {before:,} rows")
print(f"Removed: {before - after:,} near-zero revenue rows (PADS placeholders + bank charges)")
print(f"After:  {after:,} rows")
print(f"\nNew minimum revenue: £{df_clean['Revenue'].min():.2f}")

Before: 779,425 rows
Removed: 18 near-zero revenue rows (PADS placeholders + bank charges)
After:  779,407 rows

New minimum revenue: £0.06


In [14]:
print("=" * 55)
print("DATA CLEANING SUMMARY")
print("=" * 55)
print(f"Raw rows:                      {1_067_371:>10,}")
print(f"Removed — duplicates:          {34_335:>10,}  (3.22%)")
print(f"Removed — cancellations:       {19_104:>10,}  (1.79%)")
print(f"Removed — stock adjustments:   {3_393:>10,}  (0.32%)")
print(f"Removed — zero/neg price:      {2_626:>10,}  (0.25%)")
print(f"Removed — null Customer ID:    {228_488:>10,}  (21.41%)")
print(f"Removed — near-zero revenue:   {18:>10,}  (0.00%)")
print("-" * 55)
total_removed = 34_335 + 19_104 + 3_393 + 2_626 + 228_488 + 18
print(f"Total rows removed:            {total_removed:>10,}")
print(f"Final clean rows:              {779_407:>10,}")
print(f"Retention rate:                {round(779_407/1_067_371*100, 2):>9}%")
print("=" * 55)
print(f"\nFinal columns: {list(df_clean.columns)}")
print(f"Customer ID dtype: {df_clean['Customer ID'].dtype}")
print(f"Revenue range: £{df_clean['Revenue'].min():.2f} — £{df_clean['Revenue'].max():,.2f}")
print(f"Total Revenue in clean dataset: £{df_clean['Revenue'].sum():,.2f}")
print(f"Unique customers: {df_clean['Customer ID'].nunique():,}")
print(f"Unique invoices:  {df_clean['Invoice'].nunique():,}")
print(f"Unique products:  {df_clean['StockCode'].nunique():,}")

DATA CLEANING SUMMARY
Raw rows:                       1,067,371
Removed — duplicates:              34,335  (3.22%)
Removed — cancellations:           19,104  (1.79%)
Removed — stock adjustments:        3,393  (0.32%)
Removed — zero/neg price:           2,626  (0.25%)
Removed — null Customer ID:       228,488  (21.41%)
Removed — near-zero revenue:           18  (0.00%)
-------------------------------------------------------
Total rows removed:               287,964
Final clean rows:                 779,407
Retention rate:                    73.02%

Final columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue']
Customer ID dtype: int32
Revenue range: £0.06 — £168,469.60
Total Revenue in clean dataset: £17,374,804.25
Unique customers: 5,878
Unique invoices:  36,969
Unique products:  4,630


In [15]:
output_path = r"C:\Customer_Revenue_Intelligence\data\processed\online_retail_clean.csv"
df_clean.to_csv(output_path, index=False)
print(f"Clean dataset saved to: {output_path}")
print(f"Rows saved: {len(df_clean):,}")
print(f"Columns saved: {list(df_clean.columns)}")

Clean dataset saved to: C:\Customer_Revenue_Intelligence\data\processed\online_retail_clean.csv
Rows saved: 779,407
Columns saved: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue']


In [16]:
import os
file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"File exists: {os.path.exists(output_path)}")
print(f"File size: {file_size:.2f} MB")

File exists: True
File size: 72.15 MB
